# Act III — What we can do with a good graph

Three graph-native algorithms from the talk — **ranking** (Personalised PageRank), **paths**
(shortest path) and **patterns** (exact subgraph matching) — plus the simple query that opens the
door. Each starts with the intuition on the recipe graph and then shows the real-world example
from the talk.

**Where the real-world results come from (all committed, nothing runs a private pipeline):**

- **Landmark law** — a public **US Supreme Court citation graph** (27,885 cases / 234,312
  citations; CourtListener + SCDB via `idc9/law-net`). REAL. `artifacts/judgements/`.
- **The eShop code graph** — Microsoft's public **eShopOnWeb** reference app, compiled into a
  typed graph (955 nodes / 2,196 edges). REAL. `artifacts/code-graph/`.
- **Tool-call savings** — our own measured agent evals; REAL but **small-n** (state n when you quote
  it). `artifacts/proving-ground/`.

Everything runs **offline**: recipe extraction replays from cache; the rest loads committed files.


## 0. A simple graph query — which recipes contain garlic?

**Principle:** *query by relationship, not just by node — and get a subgraph back.*

The graph-database query and its relational equivalent, for comparison:

```cypher
// graph — walk the relationship
MATCH (r:Recipe)-[:CONTAINS]->(i:Ingredient)
WHERE i.name = 'garlic'
RETURN r, i
```

```sql
-- relational — join through the link table
SELECT r.title
FROM recipes r
JOIN recipe_ingredients ri ON ri.recipe_id = r.id
JOIN ingredients      i  ON i.id  = ri.ingredient_id
WHERE i.name = 'garlic';
```

One hop is already two joins. Imagine 5, 10, 20 hops — that's where graph structures inherently
excel. In NetworkX the same query is a neighbour lookup:


In [ ]:
from graphtools.data import load_hero_texts
from graphtools.extract import extract_recipe
from graphtools.graph import build_graph

# Hero recipes -> extracted records (OFFLINE replay) -> graph with entities MATCHED (Act II).
recipes = [extract_recipe(t) for _, t in load_hero_texts()]
g = build_graph(recipes, normalise=True)
print(f"recipe graph: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges\n")

seed = "ingredient:garlic"
with_garlic = sorted({u for u, _v, d in g.in_edges(seed, data=True) if d["rel"] == "CONTAINS"})
print(f"recipes that CONTAIN garlic ({len(with_garlic)}):")
for r in with_garlic:
    others = sorted(
        v.split(":", 1)[1]
        for _u, v, d in g.out_edges(r, data=True)
        if d["rel"] == "CONTAINS" and v != seed
    )
    print(f"  {g.nodes[r]['label']:28} + {len(others)} other ingredients: {', '.join(others[:6])}{', ...' if len(others) > 6 else ''}")
print("\nWhat came back isn't a row — it's a subgraph: garlic, the recipes that use it, their other ingredients.")


## 1. Personalised PageRank — which nodes matter *from here*?

**Plainly:** pick a node, wander the edges, and every so often teleport back to where you
started. The nodes you land on most often are the ones most strongly related to your start.

That teleport-home step is the *personalised* part. Vanilla PageRank (Brin & Page, 1998)
teleports to a *random* node, so it measures global popularity; PPR teleports back to your
**seed**, so it measures relevance *to the seed*. Same graph, one swap:


In [ ]:
import networkx as nx
from graphtools.algos import ppr   # nx.pagerank(personalization={seed: 1.0}) on the undirected view

ug = g.to_undirected()
seed = "ingredient:garlic"

def show(ranked, n=8):
    for node, score in [(x, s) for x, s in ranked if g.nodes[x]["kind"] in ("recipe", "ingredient")][:n]:
        print(f"   {score:.4f}  {node}")

print("GLOBAL PageRank — the popular hubs, the same for every question:")
show(sorted(nx.pagerank(ug).items(), key=lambda kv: kv[1], reverse=True))

print(f"\nPERSONALISED PageRank seeded on {seed!r} — what's related to garlic:")
show(ppr(g, [seed], top=g.number_of_nodes()))


It looks obvious on a small graph. PPR really pays off when the graph is dense and messy and it
*isn't* easy to see which nodes matter.

### PPR in the wild — landmark law (REAL)

Seed PPR on one *routine* 2013 Supreme Court case, **Kansas v. Cheever**, and walk the citation
graph. The top of the ranking is the handful of cases it cites directly — unremarkable. But at
**#8 of 27,885** sits **Miranda v. Arizona (1966)** — the "right to remain silent" landmark, which
Cheever never cites. It surfaces purely through structure, two citation hops out, and the chain
that connects them is an *auditable why* a text search would never give you.


In [ ]:
from IPython.display import Image
from graphtools.bench import load_benchmark

ppr_law = load_benchmark("ppr_landmark", source="judgements")
print(ppr_law["_provenance"], "\n")
print(f"  seed (a routine case):  {ppr_law['seed']}")
print(f"  landmark surfaced:      {ppr_law['landmark']}  ({ppr_law['hops']} citation hops away)")
print("\n  the citation chain — the auditable why:")
for i, case in enumerate(ppr_law["citation_path"]):
    print(("      " if i == 0 else "   -> ") + case)

Image(filename="../artifacts/judgements/figures/fig_ppr_path.png")


## 2. Shortest path — how does A relate to B, and through what?

**Principle:** when you know both ends but not the relationship, the most direct route between
them *is* the explanation.

Recipes aren't a great example for this one, so here is the real one from the talk: the
**eShop code graph** (Microsoft's eShopOnWeb, compiled into a typed graph of classes, methods,
`calls`, `implements`, `contains`… edges). The question: *"checkout broke after we changed the
Basket constructor — how are they even connected?"*


In [ ]:
from pathlib import Path
import networkx as nx

code_graph = nx.read_graphml(Path("..") / "artifacts" / "code-graph" / "eshop-code-graph.graphml")
PFX = "Microsoft.eShopWeb."
short = lambda n: n.replace(PFX, "")

print(f"eShop code graph: {code_graph.number_of_nodes()} nodes, {code_graph.number_of_edges()} edges")
rels = {}
for _u, _v, d in code_graph.edges(data=True):
    rels[d["rel"]] = rels.get(d["rel"], 0) + 1
print("edge types:", dict(sorted(rels.items(), key=lambda kv: -kv[1])))

# Walk only the `calls` edges: "what does checkout end up calling?"
calls = nx.DiGraph((u, v) for u, v, d in code_graph.edges(data=True) if d["rel"] == "calls")

source = PFX + "Web.Pages.Basket.CheckoutModel.OnPost"                    # the symptom
target = PFX + "ApplicationCore.Entities.BasketAggregate.Basket..ctor"   # the constructor we touched
path = nx.shortest_path(calls, source, target)

print(f"\nshortest `calls` path — {len(path) - 1} hops:")
for i, n in enumerate(path):
    print(("   " if i == 0 else "   -> ") + f"{short(n)}   [{code_graph.nodes[n].get('kind')}]")
print("\nThat chain (symbols, source text, or a summary of it) is the context you hand the agent.")


In [ ]:
# Variants you'll reach for: several routes (k shortest), a path *through* a given node, or the
# cheapest path when edges carry weights.
from itertools import islice

k_paths = list(islice(nx.shortest_simple_paths(calls, source, target), 3))
print(f"k shortest paths found: {len(k_paths)}  (lengths {[len(p) - 1 for p in k_paths]})")

via = PFX + "Web.Services.BasketViewModelService.GetOrCreateBasketForUser"
through = nx.shortest_path(calls, source, via)[:-1] + nx.shortest_path(calls, via, target)
print(f"path constrained to pass through {short(via)}: {len(through) - 1} hops")

Image(filename="../artifacts/code-graph/shortest_path.png")


### What it bought us — fewer tool calls (REAL, small-n)

Retrieving that sub-graph as context means the agent doesn't have to discover the intermediate
nodes itself with symbol lookups or vector search. In our own evaluation on the eShop (.NET)
codebase, graph-navigated code search needed **40–50% fewer tool calls per task (mean 45%,
n = 2)** to reach the same answer as a grep-based agent — with the same accuracy. On a large
PowerShell repo the saving was **56–75%** (mean 68%, n = 2).

**Read the bounds before you quote it:** *n* is two tasks per repo, one run per arm. The claim is
*efficiency, not accuracy* — both approaches found the right file every time; the graph just got
there in fewer steps. Token savings are smaller than tool-call savings (~11% on eShop, ~20% on the
PowerShell cross-project task: 59k → 47k tokens).


In [ ]:
from collections import Counter

cost = load_benchmark("cost_collapse", source="proving-ground")
print(cost["headline"], "\n")
for repo, s in cost["summary"].items():
    print(f"  {repo:11} grep {s['mean_tool_uses_grep']:>5} -> graph {s['mean_tool_uses_graph']:>4} mean tool-uses   ({s['tool_use_reduction']} fewer)")
print(f"\n  {cost['summary']['powershell']['cross_project_headline']}")

tasks_per_repo = Counter(t["repo"] for t in cost["tasks"] if t["arm"] == "grep")
print(f"  tasks per repo: {dict(tasks_per_repo)}  — small-n: a measured proof of the mechanism, not a population estimate")

Image(filename="../artifacts/proving-ground/cost_collapse.png")


## 3. Exact subgraph matching — find this *shape*, not this keyword

**Principle:** sometimes you don't know any node up front — only the shape of what you're looking
for. The query *is* a small graph, and matching finds every place it occurs.

Here we look for the **decorator pattern** in the eShop code graph: a class that wraps another
class (calls it) where **both implement the same interface** — the shape of a caching, logging or
telemetry wrapper. In a graph database:

```cypher
MATCH (cache:Class)-[:WRAPS]->(impl:Class),
      (cache)-[:IMPLEMENTS]->(i:Interface),
      (impl)-[:IMPLEMENTS]->(i)
RETURN cache, impl, i
```

Same shape in NetworkX (VF2 matcher), typed on node kind and edge relationship:


In [ ]:
from graphtools.algos import match_subgraph

# Project the multigraph to a simple DiGraph that remembers every relationship between two nodes.
typed = nx.DiGraph()
for u, v, d in code_graph.edges(data=True):
    typed.add_node(u, kind=code_graph.nodes[u].get("kind"))
    typed.add_node(v, kind=code_graph.nodes[v].get("kind"))
    typed.add_edge(u, v)
    typed[u][v].setdefault("rels", set()).add(d["rel"])

# The pattern: three variables and three edge conditions. No symbol names anywhere.
pattern = nx.DiGraph()
pattern.add_node("cache", kind="class")
pattern.add_node("impl", kind="class")
pattern.add_node("iface", kind="interface")
pattern.add_edge("cache", "impl", rel="calls")        # the wrapper calls its target
pattern.add_edge("cache", "iface", rel="implements")  # ... and both implement
pattern.add_edge("impl", "iface", rel="implements")   # ... the same interface

same_kind = lambda a, b: a.get("kind") == b.get("kind")
has_rel = lambda a, b: b["rel"] in a["rels"]

matches = match_subgraph(typed, pattern, node_match=same_kind, edge_match=has_rel)
print(f"{len(matches)} match(es) for the decorator shape in {typed.number_of_nodes()} nodes:\n")
for m in matches:
    for var in ("cache", "impl", "iface"):
        print(f"   {var:6} = {short(m[var])}")

Image(filename="../artifacts/code-graph/decorator_subgraph.png")


Boom — `CachedCatalogViewModelService` wraps `CatalogViewModelService`, both behind
`ICatalogViewModelService`. If we'd known we were looking for caching classes we could have
searched on the name; but for anti-patterns, security issues, suspicious transaction shapes or
legal arguments in a big corpus, being able to search for the *shape* without knowing the instance
is the enabling trick. (Against the compiler's own view of the codebase this match is
all-and-only — precision 1.0 / recall 1.0; see `artifacts/code-graph/bench.json`.)

## What comes next?

We toured **paths**, **ranking** and **patterns**. We skipped the classic flow / cost / search
algorithms, and didn't get to **prediction**, **similarity** or **clustering** — those edge into
graph RAG and schemaless-graph territory the talk deliberately stayed out of. Pointers, papers and
the algorithm variants mentioned above are in [`FURTHER-READING.md`](../../FURTHER-READING.md).

**Now you try:**

- Seed PPR on a recipe instead of an ingredient (`"recipe:lamb tagine"`) — what does "related"
  mean now?
- Change the shortest-path `source` to another page's handler (try `grep -i onget` over
  `code_graph.nodes`) and see which service layer it lands in.
- Write a pattern for *"a class that implements two different interfaces"* and count the matches.
